In [1]:
import os

import numpy as np
import pandas as pd
import pytorch_lightning as pl
import scanpy as sc
import torch

import T_perturb
from T_perturb.Dataloaders.datamodule import CellGenDataModule
from T_perturb.Perturb.trainer import PerturberTrainer

from T_perturb.src.utils import label_encoder
from T_perturb.tests.test_cellgen_training import dummy_dataset
from T_perturb.tests.test_countdecoder_training import dummy_cell_gene_matrix

/lustre/scratch126/cellgen/team361/kl11/t_generative/.cellgen_4096/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
seed_no = 42
pl.seed_everything(seed_no)
torch.manual_seed(seed_no)

Seed set to 42


In [3]:
if os.getcwd().split('/')[-1] != 'healthy_imm_expr':
    # set working directory to root of repository
    os.chdir('/lustre/scratch126/cellgen/team361/kl11/t_generative/')

In [4]:
tgt_vocab_size = 100  # +1 for padding token
num_samples = 100
num_genes = 100
max_seq_length = 50
n_total_tps = 2
num_samples = 100
batch_size = 4
pred_tps = [1, 2]
context_tps = [1, 2]
d_model = 12

genes_to_perturb = [70]
perturbation_token = 1


In [5]:
src_counts = dummy_cell_gene_matrix(
    num_cells=num_samples,
    num_genes=num_genes,
)
src_dataset = dummy_dataset(
    max_len=max_seq_length,
    vocab_size=tgt_vocab_size,
    num_samples=100,
)
tgt_counts_dict = dummy_cell_gene_matrix(
    num_cells=num_samples,
    num_genes=num_genes,
    total_time_steps=n_total_tps,
)
tgt_datasets = dummy_dataset(
    max_len=max_seq_length,
    vocab_size=tgt_vocab_size,
    num_samples=100,
    total_time_steps=n_total_tps,
)

In [6]:
# check if cuda is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [14]:
import importlib
import T_perturb.src.utils
importlib.reload(T_perturb.Perturb.trainer)
importlib.reload(T_perturb.src.utils)
from T_perturb.Perturb.trainer import PerturberTrainer

trainer_params = {
    'tgt_vocab_size': tgt_vocab_size,
    'd_model': d_model,
    'num_heads': 4,
    'num_layers': 1,
    'd_ff': 8,
    'max_seq_length': max_seq_length + 10,
    'dropout': 0.0,
    'pred_tps': pred_tps,
    'context_tps': context_tps,
    'n_total_tps': n_total_tps,
    'precision': 'high',
    'mask_scheduler': 'pow',
    'output_dir': './T_perturb/T_perturb/tests/res',
    'encoder': 'Transformer_encoder',
    'var_list': None,
    'genes_to_perturb': genes_to_perturb,
    'perturbation_token': perturbation_token,
    'perturbation_mode': 'generate',
    'context_mode': True,
    'temperature':1.5,
    'iterations': 19,
    'sequence_length': max_seq_length - 10,
    'pos_encoding_mode': 'time_pos_sin',
    'return_attn': False,
    'mask_scheduler': 'cosine',
}
decoder_module = PerturberTrainer(
    **trainer_params
)
data_module = CellGenDataModule(
    batch_size=batch_size,
    src_counts=src_counts,
    tgt_counts_dict=tgt_counts_dict,
    src_dataset=src_dataset,
    tgt_datasets=tgt_datasets,
    num_workers=1,
    pred_tps=pred_tps,
    context_tps=context_tps,
    n_total_tps=n_total_tps,
    split=False,
    max_len=max_seq_length,
)


Using NVIDIA A100-SXM4-80GB for training
Set float32_matmul_precision to high
cls_token_1 tensor([100])
cls_token_2 tensor([101])
Start perturbation ...
- Perturbation mode: generate
- Perturbation sequence: None
- Perturbing genes: [70]
- Replace with token: 1

Start datamodule


In [15]:
accelerator = 'gpu' if torch.cuda.is_available() else 'cpu'
trainer = pl.Trainer(
    limit_test_batches=1,  # Limit to a single batch for quick testing
    logger=False,
    accelerator=accelerator,
    devices=1,  # inference only on one gpu
)
trainer.test(
    decoder_module, 
    data_module,
    )


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
`Trainer(limit_test_batches=1)` was configured so 1 batch will be used.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing DataLoader 0:   0%|          | 0/1 [00:00<?, ?it/s]{'src_input_ids': tensor([[71, 19, 15, 96, 44, 18, 34, 55, 59, 62, 82, 65, 81, 83, 42, 90, 87, 39,
         98, 80, 56, 28, 85, 95, 73,  9, 32,  2,  5, 63, 68, 30, 45, 51, 37, 77,
         47, 33, 79, 46,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [61, 20, 42, 24, 55, 29, 28, 15, 69, 62, 67, 49, 58, 98, 85, 54, 76, 32,
         95, 80, 96, 14, 43, 89, 82, 23, 63, 31, 47, 74, 53,  3, 70, 92, 35, 52,
         38, 84, 37, 71,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 5, 65, 91, 50, 57, 82, 16, 95, 34, 27,  2, 81, 46, 51, 38, 70, 85, 90,
         58, 78,  4, 87, 17, 41, 49, 96, 79, 68, 75, 19, 40, 94, 61, 60, 74, 62,
         44, 37, 97, 21,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [67, 24, 20,  9, 87, 42, 33, 16, 32, 30, 99, 28, 55, 61, 29, 53, 38, 73,
         58, 15, 82, 50, 78, 34,  7, 94, 18, 23, 71, 17, 37, 68, 11, 83, 31, 57,
         63, 47, 62, 22,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0]],
       dev

RuntimeError: No active exception to reraise